# DR-KTE Benchmarks

In [ ]:
!git clone https://github.com/RobinLmn/deep-kte code/

import sys
sys.path.insert(0, "code/deep-kte/src")

In [ ]:
import numpy as np

from deep_kernel import EstimatedLoggingPolicy, GaussianKernel, LinearKernel, get_cme_nuisance_estimator, mmd2_dr_kpt_stat, test_pseudo_features, train_deep_kernel

from causal_medmnist import Scenario
from causal_medmnist.datasets import REGISTRY

In [ ]:
n_samples = 1000
num_repetitions = 200


def generate(scenario, n, rng, split):
    sample = scenario.generate(n, seed=int(rng.integers(0, 2**32)), split=split, replace=True)
    return sample.X.astype(np.float32), sample.A.astype(int), sample.Y


def dr_ate(X, A, Y):
    return LinearKernel()


def dr_kte(X, A, Y):
    return GaussianKernel()


def dr_kte_opt(X, A, Y):
    kernel, _, _ = train_deep_kernel(X, A, Y, "bandwidth", epochs=1000, lr=1e-2)
    return kernel.eval()


def dr_kte_deep(X, A, Y):
    kernel, _, _ = train_deep_kernel(X, A, Y, "cnn", epochs=1000, lr=1e-2)
    return kernel.eval()


def run_test(dataset, n_samples, scale, make_kernel, rng):
    scenario = Scenario(dataset, effect_strength=scale)
    n_train = int(n_samples * 0.5)

    X, A, Y = generate(scenario, n_train, rng, "train")
    cme_estimator = get_cme_nuisance_estimator(X, A)
    pi_estimator = EstimatedLoggingPolicy(X, A)
    kernel = make_kernel(X, A, Y)

    rejections = []
    for _ in range(num_repetitions):
        X_te, A_te, Y_te = generate(scenario, n_samples - n_train, rng, "val")
        features = test_pseudo_features(cme_estimator, pi_estimator, X_te, A_te)
        pval, _ = mmd2_dr_kpt_stat(Y, Y_te, features, kernel)
        rejections.append(float(pval < 0.05))

    return np.mean(rejections)


def run(scale, make_kernel, rng):
    for dataset in sorted(REGISTRY):
        result = run_test(dataset, n_samples, scale, make_kernel, rng)
        print(f"[{dataset}][N={n_samples}][scale={scale}][{make_kernel.__name__}] {result}")

In [ ]:
rng = np.random.default_rng(0)

run(scale=0.6, make_kernel=dr_ate, rng=rng)
run(scale=0.6, make_kernel=dr_kte, rng=rng)
run(scale=0.6, make_kernel=dr_kte_opt, rng=rng)
run(scale=0.6, make_kernel=dr_kte_deep, rng=rng)

run(scale=0.0, make_kernel=dr_ate, rng=rng)
run(scale=0.0, make_kernel=dr_kte, rng=rng)
run(scale=0.0, make_kernel=dr_kte_opt, rng=rng)
run(scale=0.0, make_kernel=dr_kte_deep, rng=rng)